In [ ]:
#Setup

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

In [ ]:
#Starter - paste the starter cells (roll number, generate_dataset, process_ticket, TEST_LINES) here
# The block below is only a stand-in; it is skipped when the real starter is loaded.
if 'generate_dataset' not in globals():
    print('WARNING: starter not found -> using a MOCK dataset. Run the real starter first!')
    import random, string
    from nltk.tokenize import sent_tokenize, word_tokenize
    from nltk.corpus import stopwords
    def generate_dataset():
        random.seed(1)
        base = [
            "The Wi-Fi isn't working since update 2.0.1. Contact john.doe@example.com or visit https://help.example.com/wifi now.",
            "Printer HP-200 failed 3 times. Call +91 98765 43210 for support. Battery at 95.5 percent.",
            "Outlook keeps crashing after installing version 10.2. Email support@corp.org immediately!",
            "Users are reporting slow network speeds. Admin reset passwords yesterday, but errors continued.",
        ]
        return base
    _stop = set(stopwords.words('english'))
    def process_ticket(t):
        sents = sent_tokenize(t)
        toks = word_tokenize(t)
        punct = [x for x in toks if all(ch in string.punctuation for ch in x)]
        stops = [x for x in toks if x.lower() in _stop]
        final = [x.lower() for x in toks if x not in punct and x.lower() not in _stop]
        return {'sentences': sents, 'tokens': toks, 'punctuation': punct, 'stopwords': stops, 'final_tokens': final}
    TEST_LINES = [
        "Email me at jane_doe@mail.co.uk or sales@test.org.",
        "Visit https://www.example.com/path?x=1 or http://test.io today.",
        "Call (555) 123-4567 or +1-555-123-4567 or 98765-43210.",
        "It isn't state-of-the-art, but Wi-Fi 6 reached 95.5 Mbps on v2.0.1.",
    ]

In [ ]:
#Tickets

# The ticket list. Adjust ONLY if your starter stores it under another name / needs your roll number.
tickets = tickets if 'tickets' in globals() else generate_dataset()
if isinstance(tickets, dict):          # in case generate_dataset returns {id: text}
    tickets = list(tickets.values())
tickets = [t if isinstance(t, str) else str(t) for t in tickets]
print(len(tickets), 'tickets')
for i, t in enumerate(tickets): print(i, '|', t)

In [ ]:
#Q1 - Preprocessing table

import re, string
import numpy as np
import pandas as pd
from collections import Counter
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

import re, string, pandas as pd, numpy as np
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize

def unpack(res):
    """Return dict(sentences, tokens, punct, stops, final) of lists from process_ticket() output."""
    out = {}
    if isinstance(res, dict):
        for k, v in res.items():
            kl = str(k).lower()
            if   'final'  in kl: out['final'] = v
            elif 'punct'  in kl: out['punct'] = v
            elif 'stop'   in kl: out['stops'] = v
            elif 'sent'   in kl: out['sentences'] = v
            elif 'unique' in kl: pass
            elif 'token'  in kl: out['tokens'] = v
    elif isinstance(res, (tuple, list)) and len(res) >= 5:
        out = dict(zip(['sentences', 'tokens', 'punct', 'stops', 'final'], res[:5]))
    need = {'sentences', 'tokens', 'punct', 'stops', 'final'}
    assert need <= set(out), f'unpack() could not find {need - set(out)} in process_ticket output -> edit unpack()'
    return {k: (list(v) if not isinstance(v, int) else v) for k, v in out.items()}

rows, processed = [], []
for i, t in enumerate(tickets):
    r = unpack(process_ticket(t))
    processed.append(r)
    n_sent = len(r['sentences']) if not isinstance(r['sentences'], int) else r['sentences']
    rows.append({'Ticket': i, 'Sentences': n_sent, 'Tokens': len(r['tokens']),
                 'Punctuation': len(r['punct']), 'Stopwords': len(r['stops']),
                 'Unique tokens': len({x.lower() for x in r['tokens']}),
                 'Final tokens': len(r['final'])})
df = pd.DataFrame(rows).set_index('Ticket')
df.loc['TOTAL'] = df.sum()      # note: 'Unique tokens' total = sum of per-ticket uniques (not global uniques)
df

In [ ]:
#Q1 - Selected ticket

SEL = 0   # <- choose the ticket to display
r = processed[SEL]
print('Ticket :', tickets[SEL], '\n')
print('Tokens           :', r['tokens'])
print('Punctuation      :', r['punct'])
print('Stopwords        :', r['stops'])
print('Final tokens     :', r['final'])

In [ ]:
#Q2 - Regex patterns on TEST_LINES

import re

PAT = {
    'words > 5 letters'       : r"\b[A-Za-z]{6,}\b",
    'numbers (decimals/versions)': r"(?<![\d.])\d+(?:\.\d+)*",
    'capitalized words'       : r"\b[A-Z][a-z]+(?:[-'][A-Za-z]+)*\b",
    'words starting with vowel': r"(?<![\w'@./-])[AEIOUaeiou][A-Za-z]*(?:['-][A-Za-z]+)*\b",
}
EMAIL = r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"
URL   = r"(?:https?://|www\.)[^\s<>\"']*[^\s<>\"'.,;:!?)]"
PHONE = (r"(?<![\w])(?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)\s?|\d{3}[\s.-]?)\d{3}[\s.-]?\d{4}(?!\d)"
         r"|(?<![\w])(?:\+?\d{1,3}[\s-])?\d{5}[\s-]?\d{5}(?!\d)")

def extract_all(text):
    return {name: re.findall(p, text) for name, p in PAT.items()}

def clean(text):
    text = re.sub(EMAIL, '<EMAIL>', text)
    text = re.sub(URL,   '<URL>',   text)
    text = re.sub(PHONE, '<PHONE>', text)
    return text

print('======== TEST_LINES ========')
for line in TEST_LINES:
    print('RAW  :', line)
    for k, v in extract_all(line).items(): print(f'  {k:28s}', v)
    print('CLEAN:', clean(line), '\n')

In [ ]:
#Q2 - Regex on tickets

print('======== TICKETS ========')
for i, t in enumerate(tickets):
    print(f'--- ticket {i}'); print('RAW  :', t)
    for k, v in extract_all(t).items(): print(f'  {k:28s}', v)
    print('CLEAN:', clean(t))

In [ ]:
#Q3 - my_tokenize on TEST_LINES

import pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer, word_tokenize

from nltk.tokenize import RegexpTokenizer

TOKEN_PATTERN = r'''(?x)
    (?:https?://|www\.)[^\s<>"']*[^\s<>"'.,;:!?)]                         # URL
  | [\w.+-]+@[\w-]+(?:\.[\w-]+)+                                         # email
  | (?<!\w)(?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)\s?|\d{3}[\s.-]?)\d{3}[\s.-]?\d{4}(?!\d)   # phone (US style)
  | (?<!\w)(?:\+?\d{1,3}[\s-])?\d{5}[\s-]?\d{5}(?!\d)                   # phone (5+5 style)
  | [vV]?\d+(?:\.\d+)+                                                     # decimals / versions
  | \d+                                                                    # integers
  | [A-Za-z]\w*(?:[-'\u2019]\w+)*                                          # words, contractions, hyphenated
  | [^\w\s]                                                               # any other single punctuation
'''
_tok = RegexpTokenizer(TOKEN_PATTERN)

def my_tokenize(text):
    return _tok.tokenize(text)

for line in TEST_LINES:
    print('TEXT   :', line)
    print('mine   :', my_tokenize(line))
    print('nltk   :', word_tokenize(line), '\n')

In [ ]:
#Q3 - my_tokenize on tickets

for i, t in enumerate(tickets):
    print(f'--- ticket {i}'); print('mine:', my_tokenize(t)); print('nltk:', word_tokenize(t))

In [ ]:
#Q3 - Comparison with word_tokenize

# Automatic comparison: tokens produced by only one of the two tokenizers
diffs = []
for src, texts in (('TEST_LINES', TEST_LINES), ('tickets', tickets)):
    for t in texts:
        a, b = my_tokenize(t), word_tokenize(t)
        only_mine = [x for x in a if x not in b]
        only_nltk = [x for x in b if x not in a]
        if only_mine or only_nltk:
            diffs.append({'source': src, 'text': t[:60] + ('...' if len(t) > 60 else ''),
                          'only my_tokenize': only_mine, 'only word_tokenize': only_nltk})
pd.set_option('display.max_colwidth', 90)
pd.DataFrame(diffs)

print()
print("Differences between my_tokenize() and word_tokenize():")
print("1. Contractions : word_tokenize splits isn't -> is + n't (Penn Treebank rule); my_tokenize keeps isn't as one token.")
print("2. Emails       : word_tokenize breaks name@domain.com at '@'; my_tokenize keeps the whole address.")
print("3. URLs         : word_tokenize splits https://site.com at ':'; my_tokenize keeps the full URL as one token.")
print("4. Phone numbers: word_tokenize splits (555) 123-4567 at the brackets/space; my_tokenize keeps it whole.")
print("5. Quotes       : word_tokenize turns \" into `` and ''; my_tokenize returns the original character.")
# (keep only the differences that actually appear in the table above)

In [ ]:
#Q4 - Stemming and lemmatization (setup)

import pandas as pd
import nltk

from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

final_all = [w for r in processed for w in r['final']]
uniq = sorted(set(final_all))
print(len(uniq), 'unique final tokens')

# --- choose the Regexp suffix from the data: count candidate suffixes in OUR vocabulary
cands = ['ing', 'ed', 'ly', 'es', 's', 'ion', 'ment']
cnt = {c: [w for w in uniq if w.endswith(c) and len(w) > len(c) + 2] for c in cands}
for c, ws in cnt.items(): print(f'{c:5s}', len(ws), ws[:8])

In [ ]:
#Q4 - Table of words with different outputs

# Pick the suffix with the most supporting words (needs >= 2). Override SUFFIX manually if you prefer.
ranked = sorted(((len(ws), c) for c, ws in cnt.items() if len(ws) >= 2), reverse=True)
SUFFIX = ranked[0][1] if ranked else 'ing'
support = cnt.get(SUFFIX, [])[:4]
print(f"Chosen suffix: '{SUFFIX}'  | evidence words from my dataset: {support}")
print(f"Justification: '{SUFFIX}' occurs in {len(cnt.get(SUFFIX, []))} of my {len(uniq)} unique tokens "
      f"(e.g. {', '.join(support[:2])}), so stripping it normalises these inflected forms.")

porter, lanc = PorterStemmer(), LancasterStemmer()
regs = RegexpStemmer(SUFFIX + '$', min=4)
wnl = WordNetLemmatizer()

def to_wn(tag):
    return {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}.get(tag[0], wordnet.NOUN)

# POS tag each word inside its ticket (context helps the tagger), keep first tag seen per word
pos_of = {}
for r in processed:
    for w, tag in nltk.pos_tag(r['final']):
        pos_of.setdefault(w, tag)
for w in uniq: pos_of.setdefault(w, nltk.pos_tag([w])[0][1])

res = pd.DataFrame({
    'word'      : uniq,
    'POS'       : [pos_of[w] for w in uniq],
    'Porter'    : [porter.stem(w) for w in uniq],
    'Lancaster' : [lanc.stem(w) for w in uniq],
    'Regexp'    : [regs.stem(w) for w in uniq],
    'Lemma(POS)': [wnl.lemmatize(w, to_wn(pos_of[w])) for w in uniq],
})
diff_tbl = res[res[['Porter', 'Lancaster', 'Regexp', 'Lemma(POS)']].nunique(axis=1) > 1].reset_index(drop=True)
print(len(diff_tbl), 'words where outputs differ')
diff_tbl

In [ ]:
#Q5 - Corpus statistics (S1-S4, TTR)

import pandas as pd
from collections import Counter

S1 = [x.lower() for r in processed for x in r['tokens']]
S2 = [x.lower() for r in processed for x in r['tokens'] if x not in r['punct']]
S3 = [x.lower() for r in processed for x in r['final']]
S4 = [porter.stem(x) for x in S3]

stat_rows = []
for name, S in (('S1 all tokens', S1), ('S2 no punctuation', S2), ('S3 final tokens', S3), ('S4 S3 + Porter', S4)):
    n, u = len(S), len(set(S))
    stat_rows.append({'Set': name, 'Total tokens': n, 'Unique tokens': u, 'TTR': round(u / n, 4) if n else 0})
stats = pd.DataFrame(stat_rows).set_index('Set'); stats

In [ ]:
#Q5.1 - Top 10 words of S1 and S3

print('Top 10 words S1:', Counter(S1).most_common(10))
print('Top 10 words S3:', Counter(S3).most_common(10))

In [ ]:
#Q5.2 - log(rank) vs log(frequency) for S3

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import matplotlib.pyplot as plt
freq = sorted(Counter(S3).values(), reverse=True)
rank = np.arange(1, len(freq) + 1)
plt.figure(figsize=(6, 4))
plt.plot(np.log(rank), np.log(freq), 'o-')
plt.xlabel('log(rank)'); plt.ylabel('log(frequency)'); plt.title('Zipf plot of S3 (final tokens)')
plt.grid(alpha=.3); plt.show()

In [ ]:
#Q5.3 - Top 5 bigrams of S1 and S3

from collections import Counter
from nltk import bigrams

from nltk import bigrams
def top_bigrams(per_ticket_tokens, k=5):
    c = Counter()
    for toks in per_ticket_tokens:               # bigrams inside each ticket only (no cross-ticket pairs)
        c.update(bigrams([x.lower() for x in toks]))
    return c.most_common(k)

print('Top 5 bigrams S1:', top_bigrams([r['tokens'] for r in processed]))
print('Top 5 bigrams S3:', top_bigrams([r['final']  for r in processed]))

In [ ]:
#Q6 - Interpretation

# (a) TTR
t = stats['TTR']
print('(a) TTR  S1=%.3f  S2=%.3f  S3=%.3f  S4=%.3f' % (t.iloc[0], t.iloc[1], t.iloc[2], t.iloc[3]))
print('    S1->S2 removes punctuation (repeated tokens like "." and ","), S2->S3 removes stopwords (very frequent, repeated),')
print('    S3->S4 merges inflected forms into one stem. Total tokens drop faster than unique types, so TTR generally rises;')
print('    in S4 unique types also drop because stems merge words (', len(set(S3)), '->', len(set(S4)), 'types).\n')

# (b) tokenizer difference example
ex_b = next((d for d in diffs if d['only my_tokenize'] and d['only word_tokenize']), diffs[0] if diffs else None)
print('(b) Example:', ex_b)
print('    word_tokenize applies Treebank rules (splits contractions/URLs/emails at punctuation); my_tokenize keeps them whole by design.\n')

# (c) undesirable stemming
by_stem = {}
for w in uniq: by_stem.setdefault(porter.stem(w), []).append(w)
over = [(s, ws) for s, ws in by_stem.items() if len({wnl.lemmatize(x) for x in ws}) > 1]
nonword = [(w, porter.stem(w)) for w in uniq if porter.stem(w) not in wordnet.words() and porter.stem(w) != w]
print('(c) Over-stemming (different words, one stem):', over[:3])
print('    Non-words produced by Porter:', nonword[:5])
print('    e.g. the stem is not a real word / unrelated words collapse together, which loses meaning.\n')

# (d) lemma vs stem
good = res[(res['Lemma(POS)'] != res['Porter']) & (res['Lemma(POS)'].isin(set(wordnet.words())))].head(5)
print('(d) Lemma keeps a valid word while the stemmer truncates it:')
print(good[['word', 'POS', 'Porter', 'Lemma(POS)']].to_string(index=False))
print('    With the POS tag, WordNet maps e.g. a verb/plural form to its dictionary base form instead of chopping letters.')

# Final answers (edit using the numbers printed above)
# (a) TTR changes because ...
# (b) word_tokenize vs my_tokenize: ...
# (c) Undesirable stemming: ...
# (d) POS-aware lemmatization: ...